In [1]:

import numpy as np

g = 9.81
rho_w = 1027
t = np.linspace(0, 50, 5001)

# Dive profile from Stage 2.1
time_nodes = [0, 2, 4, 6, 11, 12, 24, 25, 42, 43, 49, 50]
depth_nodes = [0, 25, 25, 15, 15, 10, 10, 6, 6, 5, 5, 0]

d = np.interp(t, time_nodes, depth_nodes)

# Wetsuit: Model B
h = np.interp(
    d,
    [0, 5, 10, 15, 20, 25],
    [6.9400, 5.0825, 4.0332, 3.4275, 2.8501, 2.2727]
)

B_suit = g * (10.32 * h / 6.94 - 2.910)

# Cylinder: cumulative gas consumption
k = rho_w * g / 101325
Q = 15 * (1 + k * d)

V_used = np.zeros_like(t)

V_used[1:] = np.cumsum(
    (Q[:-1] + Q[1:]) / 2 * np.diff(t)
)

B_cyl0 = (
    3.4 * 4.44822
    - 1.29 * g * 2191.72 * (2500 / 3000) / 1000
)

B_cyl = B_cyl0 + 1.29 * g * V_used / 1000

# Body and BCD
B_body = g * 75 * (rho_w / 985 - 1)
B_bcd0 = 1.75 * 4.44822
u_max = 150 - B_bcd0

# Stage 3: Combined buoyancy
B_base = B_body + B_suit + B_cyl + B_bcd0

m_lower = max(0, np.max(B_base) / g)
m_upper = np.min((B_base + u_max) / g)

print("Feasible interval:", m_lower, m_upper)

if m_lower <= m_upper:
    print("Minimum ballast:", round(m_lower, 2), "kg")

    u_required = m_lower * g - B_base

    print(
        "Maximum BCD adjustment:",
        round(np.max(u_required), 2),
        "N"
    )

else:
    print("No feasible ballast mass")


Feasible interval: 12.45981205793545 18.230454076694638
Minimum ballast: 12.46 kg
Maximum BCD adjustment: 85.61 N


In [2]:

def optimize_ballast(B_base, u_max, g=9.81):
    """Compute feasible ballast interval and BCD demand."""

    m_lower = max(0.0, float(np.max(B_base) / g))
    m_upper = float(np.min((B_base + u_max) / g))

    feasible = m_lower <= m_upper + 1e-9

    if not feasible:
        return {
            "feasible": False,
            "m_lower": m_lower,
            "m_upper": m_upper
        }

    u_required = m_lower * g - B_base

    return {
        "feasible": True,
        "m_lower": m_lower,
        "m_upper": m_upper,
        "max_bcd_adjustment": float(np.max(u_required)),
        "u_required": u_required
    }


# Evaluate the baseline scenario
result = optimize_ballast(B_base, u_max)

print("Feasible:", result["feasible"])
print("Minimum ballast:", round(result["m_lower"], 2), "kg")
print("Upper bound:", round(result["m_upper"], 2), "kg")
print(
    "Maximum BCD adjustment:",
    round(result["max_bcd_adjustment"], 2),
    "N"
)


Feasible: True
Minimum ballast: 12.46 kg
Upper bound: 18.23 kg
Maximum BCD adjustment: 85.61 N
